第三节，openai function in Langchain

In [37]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")

In [38]:
from typing import List
from pydantic import BaseModel, Field

In [39]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [40]:
foo = User(name="John", age=30, email="john@example.com")

In [41]:
foo.name

'John'

In [42]:
foo.age

30

In [43]:
foo.email

'john@example.com'

In [44]:
foo = User(name="John", age="30", email="john@example.com")

In [45]:
foo.age

'30'

In [46]:
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [47]:
foo = pUser(name="John", age=30, email="john@example.com")

In [48]:
foo.age

30

In [49]:
foo

pUser(name='John', age=30, email='john@example.com')

In [50]:
foo = pUser(name="John", age=30, email="john@example.com")

In [51]:
foo.age

30

In [52]:
class WeatherSearch(BaseModel):
    location: str = Field(description="The location to search for weather information")

In [53]:
from langchain_core.utils.function_calling import convert_to_openai_function

In [54]:
weather_search_function = convert_to_openai_function(WeatherSearch)

In [55]:
weather_search_function

{'name': 'WeatherSearch',
 'description': '',
 'parameters': {'properties': {'location': {'description': 'The location to search for weather information',
    'type': 'string'}},
  'required': ['location'],
  'type': 'object'}}

In [56]:
# 使用 langchain_openai（支持 bind_tools），不要用已废弃的 langchain_classic
from langchain_openai import ChatOpenAI

In [57]:
model = ChatOpenAI(
    model="LongCat-Flash-Chat",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"  # 兼容 OpenAI 的第三方 API
)

In [60]:
# 正确写法：先用 bind_tools 绑定工具，再 invoke（不能把 tools 直接传给 invoke）
from pprint import pprint

model_with_tools = model.bind_tools([weather_search_function])
msg = model_with_tools.invoke("What is the weather in San Francisco?")

# 用 pprint 带换行打印，避免整段挤在一行
pprint(msg.model_dump())

{'additional_kwargs': {'refusal': None},
 'content': '',
 'id': 'lc_run--019cfffb-abc1-7371-9ef5-c186cbbc86d7-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'tool_calls',
                       'id': 'ae18529111294031acd18eec2544ee7b',
                       'logprobs': None,
                       'model_name': 'longcat-flash-chatai-api',
                       'model_provider': 'openai',
                       'system_fingerprint': None,
                       'token_usage': {'cache_read_tokens': 0,
                                       'cache_write_tokens': 0,
                                       'cached_tokens': 0,
                                       'completion_tokens': 22,
                                       'completion_tokens_details': None,
                                       'input_tokens': 0,
                                       'output_tokens': 0,
                                       'prompt_tokens': 218,
               